# Kandinsky V2A — Longer Video, Speed and Quality Pilot

## What we learned
The short **video-led control** prompt was the clear winner. This pilot keeps that simple prompt style and tests it on a longer, more complex video.

## What we compare
- The same three five-second windows from one longer video
- Kandinsky **Full Lite** versus **Lite Distill**
- Same prompt and paired seed for each matching window
- One model process at a time; each pipeline stays loaded while its three windows are generated

The main trade-off is **sound quality versus generation time**. We also listen to the concatenated 15-second outputs for scene consistency and window seams. No training, prompt variants, denoising, or audio effects are used, so the comparison stays interpretable.

Kandinsky 6 is designed around five-second clips. Longer scenes are therefore tested as sequential five-second windows, not as one unsupported long generation. Set window start times to match the key events in your video. The duration passed to Kandinsky is the integer `5`.

## Step 1 — Select the video and its windows

Enter a short factual description of the visible sound events. Set `VIDEO_PATH` if the selected video is not the only video in `/content`. The default windows cover 0–5, 5–10, and 10–15 seconds; change them if the important action happens elsewhere.

Use the **same event description** for all windows so the video remains the main source of context. The description can name the scene’s recurring activity, for example: “Three people play volleyball in a park.”

In [ ]:
import csv
import gc
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path
from IPython.display import Audio, FileLink, Markdown, display

if sys.version_info < (3, 13):
    raise RuntimeError(f"Python 3.13+ is required. Found: {sys.version}")

def run_checked(command, cwd=None, label="command", env=None):
    result = subprocess.run(command, cwd=cwd, env=env, capture_output=True, text=True)
    if result.returncode:
        print(result.stdout[-2500:])
        print(result.stderr[-5000:])
        raise RuntimeError(f"{label} failed.")
    return result

if not shutil.which("nvidia-smi"):
    raise RuntimeError("No GPU runtime. Select a Colab GPU and reconnect.")
gpu = run_checked([
    "nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
    "--format=csv,noheader,nounits"
], label="inspect GPU")
gpu_fields = next(csv.reader([gpu.stdout.strip()]))
GPU_NAME = gpu_fields[0].strip()
GPU_MEMORY_MIB = int(gpu_fields[1])
GPU_COMPUTE = gpu_fields[2].strip() if len(gpu_fields) > 2 else "unknown"
print(f"GPU: {GPU_NAME} | {GPU_MEMORY_MIB:,} MiB | compute {GPU_COMPUTE}")
if GPU_MEMORY_MIB < 76000:
    raise RuntimeError("Full Lite needs an 80 GB-class GPU for this comparison.")

# Set a path such as /content/my_long_video.mp4 if there are multiple candidates.
VIDEO_PATH = None
# Short, factual English description of the recurring/main visible action.
EVENT_DESCRIPTION = ""
# Pick three five-second windows from the longer scene; starts must be whole seconds.
WINDOW_STARTS_S = [0, 5, 10]
DURATION_S = 5  # Keep this an integer: Kandinsky uses it to calculate frame counts.
SEED_BASE = 42

VIDEO_SUFFIXES = {".mp4", ".mov", ".webm", ".mkv"}
video_candidates = sorted(
    p for p in Path("/content").iterdir()
    if p.is_file() and p.suffix.lower() in VIDEO_SUFFIXES
    and not p.name.lower().startswith(("kandinsky_v2a_", "candidate_", "output_"))
)
if VIDEO_PATH is None:
    if len(video_candidates) == 1:
        VIDEO_PATH = video_candidates[0]
    else:
        print("Video files found in /content:")
        for candidate in video_candidates:
            print(" -", candidate)
        raise RuntimeError("Set VIDEO_PATH to the selected video's exact Colab path, then rerun this cell.")

VIDEO_PATH = Path(VIDEO_PATH)
if not VIDEO_PATH.is_file():
    raise FileNotFoundError(f"Video not found: {VIDEO_PATH}")
if not EVENT_DESCRIPTION.strip():
    raise ValueError("Fill EVENT_DESCRIPTION with a short English description of the visible action.")
if not WINDOW_STARTS_S or any(not isinstance(start, int) for start in WINDOW_STARTS_S):
    raise ValueError("WINDOW_STARTS_S must be a non-empty list of whole seconds.")

source_probe = run_checked([
    "ffprobe", "-v", "error", "-show_entries", "format=duration",
    "-of", "default=noprint_wrappers=1:nokey=1", str(VIDEO_PATH)
], label="inspect source duration")
SOURCE_DURATION_S = float(source_probe.stdout.strip())
if any(start < 0 or start + DURATION_S > SOURCE_DURATION_S for start in WINDOW_STARTS_S):
    raise ValueError(f"Every selected window must fit inside the {SOURCE_DURATION_S:.2f}s source video.")

scratch = Path("/content/local-scratch")
storage_options = [path for path in (Path("/content"), scratch) if path.exists()]
CACHE_ROOT = max(storage_options, key=lambda path: shutil.disk_usage(path).free)
FREE_GIB = shutil.disk_usage(CACHE_ROOT).free / (1024**3)
if FREE_GIB < 65:
    raise RuntimeError(f"Allow at least 65 GiB free for both checkpoints and the environment. Found {FREE_GIB:.1f} GiB.")

WORK = CACHE_ROOT / "kandinsky_long_clip_speed_quality_pilot"
INPUT_DIR = WORK / "input_windows"
OUTPUT_DIR = WORK / "outputs"
for folder in (INPUT_DIR, OUTPUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

WINDOWS = []
for index, start_s in enumerate(WINDOW_STARTS_S, start=1):
    window_path = INPUT_DIR / f"window_{index:02d}_{start_s:03d}s_silent.mp4"
    run_checked([
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
        "-ss", str(start_s), "-i", str(VIDEO_PATH), "-t", str(DURATION_S),
        "-map", "0:v:0", "-an",
        "-vf", "fps=24,scale='min(960,iw)':-2",
        "-c:v", "libx264", "-preset", "veryfast", "-crf", "25",
        "-pix_fmt", "yuv420p", str(window_path)
    ], label=f"prepare window {index}")
    probe = run_checked([
        "ffprobe", "-v", "error", "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1", str(window_path)
    ], label=f"verify window {index}")
    duration = float(probe.stdout.strip())
    if abs(duration - DURATION_S) > 0.05:
        raise RuntimeError(f"Window {index} is {duration:.3f}s; expected five seconds.")
    WINDOWS.append({
        "index": index, "start_s": start_s,
        "path": str(window_path), "sha256": sha256_file(window_path)
    })
    print(f"Window {index}: {start_s}–{start_s + DURATION_S}s | SHA-256 {WINDOWS[-1]['sha256']}")

SOURCE_SHA256 = sha256_file(VIDEO_PATH)
print(f"Source: {VIDEO_PATH.name} | {SOURCE_DURATION_S:.2f}s | SHA-256 {SOURCE_SHA256}")
print(f"Free disk: {FREE_GIB:.1f} GiB on {CACHE_ROOT}")
print(f"Prepared {len(WINDOWS)} windows. No source audio is passed to the model.")

## Step 2 — Install the pinned environment and checkpoint weights

The notebook reuses the same Kandinsky source revision as the last pilot. Both checkpoints are downloaded before generation. If the Full Lite weights are already cached under this notebook’s Kandinsky cache, the download step should reuse them.

In [ ]:
SOURCE_COMMIT = "7ceeefad7fb625011f7db602c43893a989f19c15"
REPO = Path("/content/kandinsky-v2a-worker-src")
KANDINSKY_HOME = CACHE_ROOT / "kandinsky-model-cache"
os.environ["KANDINSKY_HOME"] = str(KANDINSKY_HOME)

if not (REPO / ".git").exists():
    run_checked([
        "git", "clone", "--filter=blob:none",
        "https://github.com/kandinskylab/kandinsky-6.git", str(REPO)
    ], label="clone pinned Kandinsky source")
run_checked(["git", "fetch", "origin", SOURCE_COMMIT], cwd=REPO, label="fetch pinned source")
run_checked(["git", "checkout", "--detach", SOURCE_COMMIT], cwd=REPO, label="checkout pinned source")

if not shutil.which("uv"):
    run_checked([sys.executable, "-m", "pip", "install", "-q", "uv"], label="install uv")
if not shutil.which("just"):
    run_checked([
        "bash", "-lc",
        "curl -fsSL https://just.systems/install.sh | bash -s -- --to /usr/local/bin"
    ], label="install just")

setup = subprocess.run(
    ["uv", "sync", "--extra", "consumer", "--inexact", "--no-dev"],
    cwd=REPO, capture_output=True, text=True
)
if setup.returncode:
    print(setup.stdout[-2500:])
    print(setup.stderr[-5000:])
    raise RuntimeError("Kandinsky dependency setup failed.")

PY = REPO / ".venv" / "bin" / "python"
if not PY.is_file():
    raise FileNotFoundError(f"Pinned environment was not created: {PY}")

run_checked(["just", "download", "lite"], cwd=REPO, label="download Full Lite")
run_checked(["just", "download", "lite-distill"], cwd=REPO, label="download Lite Distill")

print("Pinned source:", SOURCE_COMMIT)
print("Worker Python:", PY)
print("Weights cache:", KANDINSKY_HOME / "weights")
print("Full Lite and Lite Distill are ready.")

## Step 3 — Generate windows with both checkpoints

The prompt stays short, using the winning video-led framing. The code runs Full Lite across all windows, exits that worker process to release its GPU memory, then runs Lite Distill across the same windows. Each checkpoint pipeline is loaded only once for its set of windows.

The seed is paired by window: the Full Lite and Distill outputs for a given window use the same seed. Timing reports separate pipeline loading from each five-second generation.

In [ ]:
PROMPT = (
    "Generate natural sound that matches the visible events in this video. "
    + EVENT_DESCRIPTION
    + " No music. No speech."
)
print("Fixed prompt:", PROMPT)

CONFIGS = {
    "full_lite": {
        "checkpoint": "lite",
        "label": "Full Lite"
    },
    "lite_distill": {
        "checkpoint": "lite-distill",
        "label": "Lite Distill"
    }
}

for key, model in CONFIGS.items():
    model_output = OUTPUT_DIR / key
    model_output.mkdir(parents=True, exist_ok=True)
    config_path = REPO / f"audio_quality_{key}.yaml"
    config_path.write_text(f"""checkpoint: {model['checkpoint']}
cache:
  mode: none
offload:
  strategy: none
compile:
  strategy: none
attention:
  engine: sdpa
text_embedder:
  quantized_qwen: false
beautifier:
  name: none
""", encoding="utf-8")

    report_path = model_output / "generation_report.json"
    driver_path = WORK / f"generate_{key}.py"
    spec_path = WORK / f"spec_{key}.json"

    driver_source = r'''import hashlib, json, sys, time, traceback
from pathlib import Path
import torch
from kandinsky import get_pipeline

spec = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
output_dir = Path(spec["output_dir"])
output_dir.mkdir(parents=True, exist_ok=True)
report_path = Path(spec["report_path"])

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

report = {
    "schema": "kandinsky_v2a_long_clip_speed_quality_pilot_v1",
    "checkpoint": spec["checkpoint"],
    "checkpoint_label": spec["checkpoint_label"],
    "source_commit": spec["source_commit"],
    "gpu": torch.cuda.get_device_name(0),
    "torch_version": torch.__version__,
    "source_video": spec["source_name"],
    "source_sha256": spec["source_sha256"],
    "prompt": spec["prompt"],
    "duration_seconds": spec["duration_seconds"],
    "windows": spec["windows"],
    "seed_base": spec["seed_base"],
    "pipeline_load_seconds": None,
    "runs": [],
    "error": None
}

def save_report():
    temp = report_path.with_suffix(".tmp")
    temp.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    temp.replace(report_path)

try:
    torch.cuda.synchronize()
    started = time.perf_counter()
    pipe = get_pipeline(spec["config"], device="cuda", attention_engine="sdpa")
    torch.cuda.synchronize()
    report["pipeline_load_seconds"] = round(time.perf_counter() - started, 2)
    print(f"{spec['checkpoint_label']} pipeline load: {report['pipeline_load_seconds']}s", flush=True)
    save_report()

    for window in spec["windows"]:
        seed = spec["seed_base"] + window["index"] - 1
        output_path = output_dir / f"window_{window['index']:02d}_{window['start_s']:03d}s_seed_{seed}.mp4"
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        started = time.perf_counter()
        pipe(
            text=spec["prompt"],
            video=window["path"],
            time_length=spec["duration_seconds"],
            seed=seed,
            save_path=str(output_path),
            show_progress=True
        )
        torch.cuda.synchronize()
        elapsed = round(time.perf_counter() - started, 2)
        if not output_path.is_file() or output_path.stat().st_size == 0:
            raise RuntimeError(f"Output was not written: {output_path}")
        item = {
            "window_index": window["index"],
            "window_start_s": window["start_s"],
            "seed": seed,
            "seconds": elapsed,
            "peak_allocated_mib": round(torch.cuda.max_memory_allocated() / (1024**2), 1),
            "peak_reserved_mib": round(torch.cuda.max_memory_reserved() / (1024**2), 1),
            "output_mp4": output_path.name,
            "output_sha256": sha256_file(output_path),
            "output_bytes": output_path.stat().st_size
        }
        report["runs"].append(item)
        save_report()
        print(f"Window {window['index']} ({window['window_start_s']}s): {elapsed}s", flush=True)
except Exception as exc:
    report["error"] = f"{type(exc).__name__}: {exc}"
    report["traceback"] = traceback.format_exc()
    save_report()
    raise
'''

    driver_path.write_text(driver_source, encoding="utf-8")
    spec = {
        "checkpoint": model["checkpoint"],
        "checkpoint_label": model["label"],
        "source_commit": SOURCE_COMMIT,
        "source_name": VIDEO_PATH.name,
        "source_sha256": SOURCE_SHA256,
        "prompt": PROMPT,
        "duration_seconds": DURATION_S,
        "seed_base": SEED_BASE,
        "windows": WINDOWS,
        "config": str(config_path),
        "output_dir": str(model_output),
        "report_path": str(report_path)
    }
    spec_path.write_text(json.dumps(spec, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"\nStarting {model['label']} for {len(WINDOWS)} windows.")
    worker = subprocess.run([str(PY), str(driver_path), str(spec_path)], cwd=REPO, env=os.environ.copy())
    if worker.returncode:
        detail = ""
        if report_path.is_file():
            saved = json.loads(report_path.read_text(encoding="utf-8"))
            detail = saved.get("traceback") or saved.get("error") or ""
        raise RuntimeError(f"{model['label']} worker failed.\n{detail}")
    del worker
    gc.collect()
    print(f"{model['label']} complete. GPU memory is released before the next checkpoint.")

## Step 4 — Listen and compare

First compare the same window across Full Lite and Lite Distill. Then listen to each concatenated 15-second WAV to judge whether scene character stays consistent across windows. The joined WAV is a plain concatenation with no crossfade, so window seams remain audible for evaluation.

In [ ]:
from IPython.display import Audio, FileLink, Markdown, display

ALL_REPORTS = {}
for key, model in CONFIGS.items():
    model_output = OUTPUT_DIR / key
    report_path = model_output / "generation_report.json"
    report = json.loads(report_path.read_text(encoding="utf-8"))
    if report.get("error"):
        raise RuntimeError(f"{model['label']} report contains an error: {report['error']}")
    ALL_REPORTS[key] = report

    wav_paths = []
    for run_info in report["runs"]:
        mp4_path = model_output / run_info["output_mp4"]
        wav_path = model_output / (Path(run_info["output_mp4"]).stem + ".wav")
        extraction = subprocess.run([
            "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
            "-i", str(mp4_path), "-map", "0:a:0", "-vn",
            "-ac", "2", "-ar", "44100", "-c:a", "pcm_s16le", str(wav_path)
        ], capture_output=True, text=True)
        if extraction.returncode:
            print(extraction.stderr[-2000:])
            raise RuntimeError(f"Audio extraction failed for {mp4_path.name}")
        run_info["output_wav"] = wav_path.name
        run_info["wav_sha256"] = sha256_file(wav_path)
        wav_paths.append(wav_path)
        display(Markdown(
            f"### {model['label']} — Window {run_info['window_index']} "
            f"({run_info['window_start_s']}–{run_info['window_start_s'] + DURATION_S}s) "
            f"— {run_info['seconds']}s"
        ))
        display(Audio(filename=str(wav_path)))

    # Concatenate the exact five-second WAVs in source-video order; no crossfade.
    concat_list = model_output / "wav_concat_list.txt"
    concat_list.write_text(
        "".join(f"file '{path.as_posix()}'\n" for path in wav_paths),
        encoding="utf-8"
    )
    joined_wav = model_output / f"{key}_joined_windows.wav"
    joined = subprocess.run([
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
        "-f", "concat", "-safe", "0", "-i", str(concat_list),
        "-c:a", "pcm_s16le", str(joined_wav)
    ], capture_output=True, text=True)
    if joined.returncode:
        print(joined.stderr[-2000:])
        raise RuntimeError(f"Could not concatenate the {model['label']} WAV windows.")
    report["joined_wav"] = joined_wav.name
    report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

    display(Markdown(f"## {model['label']} — concatenated long-scene listen"))
    display(Audio(filename=str(joined_wav)))

    timing = [run["seconds"] for run in report["runs"]]
    print(
        f"{model['label']}: load {report['pipeline_load_seconds']}s | "
        f"window median {sorted(timing)[len(timing)//2]}s | "
        f"total generation {sum(timing)}s"
    )

## Step 5 — Save a listening report

Rate each candidate using the same criteria. The timing and GPU memory measurements are saved alongside hashes. The ZIP intentionally excludes the input video.

In [ ]:
REVIEW_PATH = WORK / "listening_ratings.json"
review = {
    "schema": "kandinsky_v2a_long_clip_ratings_v1",
    "input_video_sha256": SOURCE_SHA256,
    "event_description": EVENT_DESCRIPTION,
    "window_starts_s": WINDOW_STARTS_S,
    "ratings": {}
}
for key, report in ALL_REPORTS.items():
    for run_info in report["runs"]:
        rating_id = f"{key}_window_{run_info['window_index']}"
        review["ratings"][rating_id] = {
            "event_match_1_to_5": None,
            "naturalness_1_to_5": None,
            "artifacts_1_to_5": None,
            "timing_fit_1_to_5": None,
            "unwanted_music_or_speech": None,
            "would_keep": None,
            "notes": ""
        }
    review["ratings"][f"{key}_joined"] = {
        "scene_consistency_1_to_5": None,
        "window_seams_1_to_5": None,
        "would_use_for_long_scene": None,
        "notes": ""
    }
REVIEW_PATH.write_text(json.dumps(review, ensure_ascii=False, indent=2), encoding="utf-8")

RESULTS_ZIP = Path("/content/kandinsky_long_clip_speed_quality_results.zip")
with zipfile.ZipFile(RESULTS_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(REVIEW_PATH, arcname="listening_ratings_template.json")
    for model_dir in OUTPUT_DIR.iterdir():
        if model_dir.is_dir():
            for path in model_dir.rglob("*"):
                if path.is_file():
                    archive.write(path, arcname=str(Path("outputs") / path.relative_to(OUTPUT_DIR)))

print("Results ZIP:", RESULTS_ZIP)
print("ZIP size: %.1f MiB" % (RESULTS_ZIP.stat().st_size / (1024**2)))
display(FileLink(str(RESULTS_ZIP)))

## How we will use the result

- If Lite Distill is much faster and keeps acceptable sound quality across the windows, it becomes the quick iteration path.
- If Full Lite remains clearly better, use it as the quality reference and test inference acceleration separately on the same five-second window.
- If both checkpoints sound weak on one event type, we should route that event to a sound-focused model rather than expect video-to-audio alone to solve every quality problem.

A published H100 80 GB benchmark for five-second generation reports lower latency for Lite Distill than Lite, but it measures the authors' synchronized text-to-video-and-audio benchmark, not this exact Colab video-to-audio workload. Our measured Colab timings and listening judgments are the decision evidence for this project. See the [official Kandinsky benchmark and checkpoint details](https://github.com/kandinskylab/kandinsky-6) and [video-to-audio limits](https://github.com/kandinskylab/kandinsky-6/blob/main/docs/generation_modes.md).